# 06_did_fixed_effects

Individual fixed-effects verification of the BMI -> hypertension pathway.

The new-onset outcome is defined so that everyone is disease-free at baseline
(pre-period outcome fixed at 0), which makes a classic 2x2 difference-in-
differences degenerate: within transformation collapses the treated:post
coefficient onto the treated group's post incidence. We therefore use a
**conditional (fixed-effects) logistic model**, which conditions on the
individual and identifies the BMI effect purely from within-person variation,
absorbing all time-invariant unobserved confounders (genetics, health
literacy, personality). This is the strongest causal control available in the
panel. A linear DID with cohort fixed effects and clustered SE is reported as
a cross-check.

In [1]:
# 06_did_fixed_effects.ipynb
# Within-person (fixed-effects) estimation of the BMI -> HTN new-onset effect.

import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import statsmodels.formula.api as smf
from statsmodels.discrete.conditional_models import ConditionalLogit

ROOT = os.path.abspath("..")
DATA = os.path.join(ROOT, "data")
TAB  = os.path.join(ROOT, "results", "tables")

T = pd.read_parquet(os.path.join(DATA, "khp_transitions.parquet"))
htn = T[T["HTN"] == 0].copy()
htn["incident"] = (htn["HTN_t1"] == 1).astype(int)
htn = htn.dropna(subset=["BMI", "age", "smoke_cur", "exer_reg"])
print("HTN at-risk observations:", len(htn))

HTN at-risk observations: 31179


In [2]:
# Conditional logit uses only individuals with >=2 observations AND
# within-person variation in the outcome (both 0 and 1 experienced).
cnt   = htn["PIDWON"].value_counts()
multi = cnt[cnt >= 2].index
hm    = htn[htn["PIDWON"].isin(multi)].copy()

var         = hm.groupby("PIDWON")["incident"].nunique()
informative = var[var > 1].index
hc = hm[hm["PIDWON"].isin(informative)].copy()

print(f"Repeated-measure individuals : {len(multi)}")
print(f"Informative (outcome varies) : {len(informative)}")
print(f"Contributing observations    : {len(hc)}")

Repeated-measure individuals : 7158
Informative (outcome varies) : 683
Contributing observations    : 2334


In [3]:
# Fixed-effects conditional logistic regression: within-person BMI effect.
y      = hc["incident"].values
X      = hc[["BMI", "smoke_cur", "exer_reg"]].astype(float)
groups = hc["PIDWON"].values

res = ConditionalLogit(y, X, groups=groups).fit(disp=0)
OR  = np.exp(res.params)
ci  = np.exp(res.conf_int())

fe_tab = pd.DataFrame({
    "Variable": ["BMI", "Current smoker", "Regular exercise"],
    "OR":     [OR["BMI"], OR["smoke_cur"], OR["exer_reg"]],
    "CI_low": [ci.loc["BMI",0], ci.loc["smoke_cur",0], ci.loc["exer_reg",0]],
    "CI_high":[ci.loc["BMI",1], ci.loc["smoke_cur",1], ci.loc["exer_reg",1]],
    "p":      [res.pvalues["BMI"], res.pvalues["smoke_cur"], res.pvalues["exer_reg"]],
}).round(3)
fe_tab.to_csv(os.path.join(TAB, "table5_fixed_effects.csv"), index=False)
print(fe_tab.to_string(index=False))
print("\nWithin-person BMI effect is not significant -> null is robust to"
      " time-invariant unobserved confounding.")

        Variable    OR  CI_low  CI_high     p
             BMI 1.038   0.952    1.131 0.403
  Current smoker 0.840   0.404    1.745 0.640
Regular exercise 1.128   0.899    1.414 0.299

Within-person BMI effect is not significant -> null is robust to time-invariant unobserved confounding.


In [4]:
# Cross-check 1: linear DID with cohort fixed effects, clustered SE.
# Reconstruct pre(t0)/post(t2) long form; treatment = >=1 kg/m2 BMI loss t0->t1.
panel = pd.read_parquet(os.path.join(DATA, "khp_panel_long.parquet"))
adult = panel[panel.age >= 19].copy()
TRIPLES = [(2019,2020,2021),(2020,2021,2022),(2021,2022,2023),(2022,2023,2024)]

rows = []
for t0, t1, t2 in TRIPLES:
    d0 = adult[adult.year==t0][["PIDWON","BMI","HTN"]]
    d1 = adult[adult.year==t1][["PIDWON","BMI","HTN"]]
    d2 = adult[adult.year==t2][["PIDWON","HTN"]]
    m = (d0.merge(d1,on="PIDWON",suffixes=("_0","_1"))
           .merge(d2,on="PIDWON").rename(columns={"HTN":"HTN_2"}))
    r = m[(m["HTN_0"]==0)&(m["HTN_1"]==0)].dropna(subset=["BMI_0","BMI_1","HTN_2"]).copy()
    r["treated"] = (r["BMI_1"]-r["BMI_0"] <= -1.0).astype(int)
    r["t0y"] = t0
    rows.append(r)
S = pd.concat(rows, ignore_index=True)

long = []
for _, r in S.iterrows():
    uid = f"{r['PIDWON']}_{r['t0y']}"
    long.append({"uid":uid,"post":0,"treated":r["treated"],"HTN":r["HTN_0"],"t0y":r["t0y"]})
    long.append({"uid":uid,"post":1,"treated":r["treated"],"HTN":r["HTN_2"],"t0y":r["t0y"]})
L = pd.DataFrame(long)

did = smf.ols("HTN ~ treated*post + C(t0y)", data=L).fit(
        cov_type="cluster", cov_kwds={"groups": L["uid"]})
b  = did.params["treated:post"]; ci = did.conf_int().loc["treated:post"]
print(f"Linear DID (treated x post) = {b*100:+.3f} pp "
      f"(95% CI {ci[0]*100:+.3f} to {ci[1]*100:+.3f}), p={did.pvalues['treated:post']:.3f}")

Linear DID (treated x post) = +0.487 pp (95% CI -0.297 to +1.272), p=0.224


In [5]:
# Cross-check 2: selection into treatment (baseline BMI differs by group).
S4 = pd.read_parquet(os.path.join(DATA, "step4_data.parquet"))
print("Baseline BMI by treatment status (self-selection check):")
print(f"  Achieved (reduced) : {S4[S4.achieved==1]['BMI_0'].mean():.2f}")
print(f"  Not achieved       : {S4[S4.achieved==0]['BMI_0'].mean():.2f}")
print("\nHigher baseline BMI among reducers explains the crude positive"
      " association (regression to the mean / health motivation).")

Baseline BMI by treatment status (self-selection check):
  Achieved (reduced) : 25.04
  Not achieved       : 23.08

Higher baseline BMI among reducers explains the crude positive association (regression to the mean / health motivation).
